# Recomendador de contenido (libros) con TF-IDF y similitud coseno
**NLP desde cero · Semana 1 · Proyecto 5: Recomendador de contenido**

**Entregable del curso:** endpoint `/recomendar` + evaluación con juicios de usuarios reales.

| Parte | Qué hace | Dónde está |
|---|---|---|
| 1 | Carga del dataset de Kaggle y muestreo de un catálogo de ~2000 libros | Secciones 1-2 |
| 2 | Preprocesamiento: minúsculas, sin tildes, sin stopwords, lematización con spaCy | Sección 3 |
| 3 | `TfidfVectorizer` (1-2 gramas, `sublinear_tf=True`, `min_df` ajustado) + similitud coseno top-N | Sección 4 |
| 4 | Evaluación: métrica proxy por géneros + encuesta con usuarios reales (Precision@K, nDCG@K) | Sección 5 |
| 5 | **Endpoint** FastAPI `/recomendar` | Sección 6 |
| 6 | **Dashboard** en el notebook (ipywidgets) y **aplicación web** Streamlit lista para ejecutar | Sección 7 |
| 7 | Mejora opcional: embeddings (sentence-transformers) para mostrar el límite de TF-IDF | Sección 8 |

**Cómo ejecutarlo en Google Colab:** `Entorno de ejecución → Ejecutar todo`. Todo lo configurable está en la celda **CONFIGURACIÓN** (sección 0).
El dataset se descarga con `kagglehub`; si no funciona, el notebook te pide subir el CSV manualmente.

## 0. Instalación y configuración

In [ ]:
%pip install -q kagglehub spacy scikit-learn pandas matplotlib joblib fastapi "uvicorn[standard]" requests ipywidgets sentence-transformers

In [ ]:
import importlib, subprocess, sys

def asegurar_modelo_spacy(nombre):
    """Descarga el modelo de spaCy solo si no está instalado."""
    try:
        importlib.import_module(nombre)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "spacy", "download", nombre])
        importlib.invalidate_caches()

# Se instalan los dos modelos porque el idioma del dataset se detecta más abajo.
asegurar_modelo_spacy("es_core_news_sm")
asegurar_modelo_spacy("en_core_web_sm")
print("Listo.")

In [ ]:
# =========================  CONFIGURACIÓN  =========================
KAGGLE_DATASET = "danielalbarracinm/books-with-isbn-of-different-genres"
RUTA_CSV = None            # Si subes el CSV a mano: "/content/mi_archivo.csv"
N_ITEMS = 2000             # Tamaño del catálogo (500-2000 según el plan)
SEMILLA = 42

IDIOMA = "auto"            # "auto", "es" o "en": idioma del TEXTO (define modelo de spaCy y stopwords)
FILTRAR_IDIOMA = None      # Si el CSV tiene columna de idioma: "Spanish", "es", "English"... (None = no filtrar)

# Columnas: se detectan solas. Si falla la detección, fuérzalas aquí. Ejemplo: {"titulo": "Book", "descripcion": "Synopsis"}
COLUMNAS_MANUAL = {}

STOPWORDS_BILINGUES = True     # El catálogo mezcla español e inglés: se quitan stopwords de ambos idiomas

TEXTO_INCLUYE_TITULO = True      # El título entra al texto que se vectoriza
TEXTO_INCLUYE_GENEROS = False    # False evita que la evaluación por géneros sea circular
MIN_LARGO_DESCRIPCION = 80       # Descarta fichas con sinopsis demasiado cortas (caracteres)

# TF-IDF
NGRAMAS = (1, 2)
MIN_DF = 2                 # Se justifica con la tabla de la sección 4.1
MAX_DF = 0.85

# Evaluación
K = 5
UMBRAL_JACCARD = 0.25      # Dos libros son "relevantes" entre sí si sus géneros tienen Jaccard >= umbral
N_CONSULTAS_PROXY = 300
N_CONSULTAS_ENCUESTA = 8   # Libros de consulta que verán los evaluadores humanos
USAR_EMBEDDINGS = True     # Sección 8 (opcional). Ponlo en False si Colab va lento.
# ===================================================================

In [ ]:
import re, time, unicodedata, threading
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
rng = np.random.default_rng(SEMILLA)

def quitar_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFKD", str(texto)) if not unicodedata.combining(c))

## 1. Carga del dataset
Dataset: *Books with ISBN of different genres* (Kaggle). La celda localiza el CSV más grande de la descarga.

In [ ]:
def localizar_archivo():
    if RUTA_CSV:
        return Path(RUTA_CSV)
    try:
        import kagglehub
        carpeta = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        candidatos = sorted(
            [p for p in carpeta.rglob("*") if p.suffix.lower() in {".csv", ".tsv", ".json", ".xlsx"}],
            key=lambda p: p.stat().st_size, reverse=True)
        if not candidatos:
            raise FileNotFoundError("La descarga no contiene CSV/JSON/XLSX")
        print("Archivos encontrados:", [f"{p.name} ({p.stat().st_size/1e6:.1f} MB)" for p in candidatos])
        return candidatos[0]
    except Exception as e:
        print(f"No se pudo descargar con kagglehub ({type(e).__name__}: {e}).")
        try:
            from google.colab import files
            print("Sube el archivo del dataset (CSV):")
            subido = files.upload()
            return Path(next(iter(subido)))
        except ImportError:
            raise FileNotFoundError("Descarga el CSV desde Kaggle y ponlo en RUTA_CSV.")

def leer_tabla(ruta):
    ruta = Path(ruta)
    if ruta.suffix.lower() == ".xlsx":
        return pd.read_excel(ruta)
    if ruta.suffix.lower() == ".json":
        return pd.read_json(ruta)
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, encoding=enc, sep=None, engine="python", on_bad_lines="skip")
        except UnicodeDecodeError:
            continue
    raise ValueError("No se pudo leer el archivo")

archivo = localizar_archivo()
bruto = leer_tabla(archivo)
print(f"\nArchivo: {archivo.name}  |  {bruto.shape[0]:,} filas x {bruto.shape[1]} columnas")
bruto.head(3)

In [ ]:
print("Columnas:", list(bruto.columns))
bruto.info(max_cols=30)

## 2. Armado del catálogo
Se detectan las columnas (título, autor, descripción, géneros, idioma), se limpian duplicados y se muestrea el catálogo.

> **Importante:** si el dataset no trae una columna de descripción/sinopsis, el recomendador solo puede apoyarse en título y géneros y la calidad baja bastante. La celda avisa si pasa.

In [ ]:
CANDIDATAS = {
    "titulo":      ["title", "titulo", "book_title", "book", "name", "book_name", "titulo_libro"],
    "autor":       ["author", "writer", "escritor", "authors", "autor", "autores", "book_author"],
    "descripcion": ["description", "descripcion", "synopsis", "sinopsis", "summary", "resumen", "plot", "blurb", "about", "overview"],
    "generos":     ["genres", "genre", "gender", "genero", "generos", "categories", "category", "categoria", "subjects", "main_genre"],
    "idioma":      ["language", "idioma", "lang", "language_code"],
    "isbn":        ["isbn", "isbn13", "isbn_13", "isbn10", "isbn_10"],
}

def norm_col(c):
    return re.sub(r"[^a-z0-9]+", "_", quitar_tildes(str(c).lower())).strip("_")

def detectar_columnas(df, manual):
    por_norm = {norm_col(c): c for c in df.columns}
    out = {}
    for campo, nombres in CANDIDATAS.items():
        if campo in manual:
            out[campo] = manual[campo]; continue
        out[campo] = next((por_norm[n] for n in nombres if n in por_norm), None)
    return out

cols = detectar_columnas(bruto, COLUMNAS_MANUAL)
print("Columnas detectadas:")
for k, v in cols.items():
    print(f"  {k:12s} -> {v}")
assert cols["titulo"], "No se detectó la columna de título: indícala en COLUMNAS_MANUAL"
if not cols["descripcion"]:
    print("\n[AVISO] No hay columna de descripción. Se usarán título y géneros; espera recomendaciones más pobres.")
    TEXTO_INCLUYE_GENEROS = True

In [ ]:
def parse_generos(x):
    if pd.isna(x):
        return []
    return [g.strip().lower() for g in re.split(r"[,;|/\[\]'\"]+", str(x)) if g.strip()]

df = pd.DataFrame({"titulo": bruto[cols["titulo"]].astype(str).str.strip()})
df["autor"] = bruto[cols["autor"]].astype(str).str.strip() if cols["autor"] else ""
df["descripcion"] = bruto[cols["descripcion"]].fillna("").astype(str).str.strip() if cols["descripcion"] else ""
df["generos_raw"] = bruto[cols["generos"]] if cols["generos"] else np.nan
df["idioma_col"] = bruto[cols["idioma"]].astype(str) if cols["idioma"] else ""
df["isbn"] = bruto[cols["isbn"]].astype(str) if cols["isbn"] else ""

# Filtro opcional por idioma del dataset
if cols["idioma"]:
    print("Idiomas en el dataset (top 10):")
    print(df["idioma_col"].value_counts().head(10).to_string())
    if FILTRAR_IDIOMA:
        antes = len(df)
        df = df[df["idioma_col"].str.contains(FILTRAR_IDIOMA, case=False, na=False)]
        print(f"\nFiltro idioma '{FILTRAR_IDIOMA}': {antes:,} -> {len(df):,} filas")

# Limpieza
df = df[df["titulo"].str.len() > 0]
if cols["descripcion"]:
    df = df[df["descripcion"].str.len() >= MIN_LARGO_DESCRIPCION]
df["_clave"] = df["titulo"].map(lambda s: re.sub(r"\W+", " ", quitar_tildes(s.lower())).strip())
df = df.drop_duplicates("_clave").drop(columns="_clave")

rotos = df["descripcion"].str.contains("#", regex=False).mean() if cols["descripcion"] else 0
print(f"Sinopsis con caracteres dañados ('#' en lugar de tildes/ñ): {rotos:.1%}  -> esas palabras se descartan al preprocesar")
print(f"\nFichas utilizables tras la limpieza: {len(df):,}")
if len(df) > N_ITEMS:
    df = df.sample(N_ITEMS, random_state=SEMILLA)
df = df.reset_index(drop=True)
df.insert(0, "id", df.index)
df["generos_lista"] = df["generos_raw"].map(parse_generos)
df["generos"] = df["generos_lista"].map(lambda g: ", ".join(g[:6]))
print(f"Catálogo final: {len(df):,} libros")
df[["id", "titulo", "autor", "generos"]].head()

In [ ]:
# Detección del idioma del texto (define modelo de spaCy y stopwords)
def detectar_idioma(textos):
    es = {"el", "la", "los", "las", "de", "que", "y", "en", "un", "una", "por", "con", "para", "del", "se"}
    en = {"the", "of", "and", "to", "in", "a", "is", "that", "with", "for", "his", "her", "he", "she"}
    cuenta_es = cuenta_en = 0
    for t in textos:
        toks = re.findall(r"[a-záéíóúñ]+", t.lower())
        cuenta_es += sum(w in es for w in toks)
        cuenta_en += sum(w in en for w in toks)
    return "es" if cuenta_es > cuenta_en else "en"

muestra_txt = (df["descripcion"] if cols["descripcion"] else df["titulo"]).head(300)
idioma_texto = detectar_idioma(muestra_txt) if IDIOMA == "auto" else IDIOMA
print(f"Idioma del texto: {idioma_texto.upper()}  ->  modelo spaCy: "
      f"{'es_core_news_sm' if idioma_texto == 'es' else 'en_core_web_sm'}")
if STOPWORDS_BILINGUES:
    print("Stopwords: español + inglés (el catálogo mezcla ambos idiomas).")
if idioma_texto == "en":
    print("(El dataset está mayormente en inglés; el pipeline es idéntico, solo cambian modelo y stopwords. "
          "Para un catálogo en español usa FILTRAR_IDIOMA o carga otro CSV.)")

In [ ]:
# Exploración rápida del catálogo
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
if cols["descripcion"]:
    df["descripcion"].str.split().str.len().clip(upper=400).plot.hist(bins=40, ax=ax[0], color="#2a7f9e")
    ax[0].set_title("Longitud de la sinopsis (palabras, recortado a 400)")
else:
    ax[0].axis("off")
todos = pd.Series([g for lst in df["generos_lista"] for g in lst])
if len(todos):
    todos.value_counts().head(12).iloc[::-1].plot.barh(ax=ax[1], color="#2a7f9e")
    ax[1].set_title("Géneros más frecuentes")
else:
    ax[1].axis("off")
plt.tight_layout(); plt.show()

## 3. Preprocesamiento (módulo `recomendador.py`)
Se escribe un módulo reutilizable. **El mismo código lo usan el notebook, la API y la aplicación Streamlit**, así que el texto de las consultas se procesa igual que el del catálogo.

Pasos del preprocesamiento, en este orden:
1. **Lematización con spaCy** (se hace primero porque el lematizador aprovecha las tildes y la forma original de la palabra).
2. Minúsculas.
3. **Eliminación de tildes** (`unicodedata`, forma NFKD).
4. **Eliminación de stopwords** (lista de spaCy, también sin tildes) y de tokens no alfabéticos o de menos de 3 letras. Como el catálogo mezcla sinopsis en español e inglés, por defecto se quitan las stopwords **de los dos idiomas** (`STOPWORDS_BILINGUES`); si no, palabras como *the*, *and* o *with* dominan la similitud.

Antes de todo, se descartan las palabras con `#`: en este dataset ese carácter reemplaza a tildes y eñes mal codificadas (*Amar#s*, *Despu#s*) y no se pueden recuperar.

In [ ]:
%%writefile recomendador.py
"""Recomendador de contenido: preprocesamiento + TF-IDF + similitud coseno."""
import re
import unicodedata

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

MODELOS_SPACY = {"es": "es_core_news_sm", "en": "en_core_web_sm"}
MAX_CHARS = 1500  # recorta sinopsis larguísimas para acelerar spaCy


def quitar_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFKD", str(texto)) if not unicodedata.combining(c))


def normalizar_titulo(s):
    return re.sub(r"\W+", " ", quitar_tildes(str(s).lower())).strip()


RE_ROTAS = re.compile(r"\w*#\w*")  # palabras con caracteres mal codificados (p. ej. "Amar#s")


def stopwords_spacy(idioma):
    if idioma == "es":
        from spacy.lang.es.stop_words import STOP_WORDS
    else:
        from spacy.lang.en.stop_words import STOP_WORDS
    return {quitar_tildes(w.lower()) for w in STOP_WORDS}


class Preprocesador:
    """minúsculas + sin tildes + sin stopwords + lematización (spaCy)."""

    def __init__(self, idioma="es", bilingue=True):
        self.idioma = idioma
        self.bilingue = bilingue
        self._nlp = None
        self._stop = None

    @property
    def nlp(self):
        if self._nlp is None:
            import spacy
            nombre = MODELOS_SPACY[self.idioma]
            try:
                self._nlp = spacy.load(nombre, disable=["parser", "ner"])
            except OSError as e:
                raise RuntimeError(f"Falta el modelo: python -m spacy download {nombre}") from e
            self._stop = {quitar_tildes(w.lower()) for w in self._nlp.Defaults.stop_words}
            if self.bilingue:
                for otro in MODELOS_SPACY:
                    self._stop |= stopwords_spacy(otro)
        return self._nlp

    def transformar(self, textos, batch_size=64, n_process=1):
        nlp = self.nlp
        salida = []
        flujo = (RE_ROTAS.sub(" ", str(t)[:MAX_CHARS]) for t in textos)
        for doc in nlp.pipe(flujo, batch_size=batch_size, n_process=n_process):
            toks = []
            for t in doc:
                if not t.is_alpha or t.is_stop:
                    continue
                lema = quitar_tildes(t.lemma_.lower())
                if len(lema) > 2 and lema not in self._stop and quitar_tildes(t.lower_) not in self._stop:
                    toks.append(lema)
            salida.append(" ".join(toks))
        return salida


class Recomendador:
    def __init__(self, idioma="es", ngram_range=(1, 2), min_df=2, max_df=0.85, bilingue=True):
        self.idioma = idioma
        self.bilingue = bilingue
        self.params = dict(ngram_range=tuple(ngram_range), min_df=min_df, max_df=max_df)
        self.pre = Preprocesador(idioma, bilingue)
        self.vec = None
        self.X = None
        self.df = None

    # ------------------------------------------------------------ entrenamiento
    def fit(self, df, col_limpio="texto_limpio", col_titulo="titulo"):
        self.df = df.reset_index(drop=True).copy()
        self.vec = TfidfVectorizer(
            ngram_range=self.params["ngram_range"],
            min_df=self.params["min_df"],
            max_df=self.params["max_df"],
            sublinear_tf=True,
            norm="l2",
        )
        self.X = self.vec.fit_transform(self.df[col_limpio].fillna(""))
        self.terminos = self.vec.get_feature_names_out()
        self.titulo_norm = self.df[col_titulo].map(normalizar_titulo).values
        return self

    # ------------------------------------------------------------ consulta por ítem
    def indices_similares(self, idx, n=5):
        """Top-N por similitud coseno. TF-IDF ya viene normalizado (L2), así que el coseno es el producto punto."""
        sims = linear_kernel(self.X[idx], self.X).ravel()
        sims[self.titulo_norm == self.titulo_norm[idx]] = -1.0  # excluye el propio libro y otras ediciones
        n = min(n, len(sims) - 1)
        top = np.argpartition(-sims, n - 1)[:n]
        top = top[np.argsort(-sims[top])]
        return top, sims[top]

    def terminos_comunes(self, i, j, k=5):
        prod = self.X[i].multiply(self.X[j]).tocoo()
        if prod.nnz == 0:
            return []
        orden = np.argsort(-prod.data)[:k]
        return [str(self.terminos[prod.col[o]]) for o in orden]

    def _tabla(self, idxs, scores, comunes):
        d = self.df
        filas = []
        for pos, (j, s) in enumerate(zip(idxs, scores)):
            filas.append({
                "id": int(d.loc[j, "id"]),
                "titulo": d.loc[j, "titulo"],
                "autor": d.loc[j, "autor"] if "autor" in d else "",
                "generos": d.loc[j, "generos"] if "generos" in d else "",
                "score": round(float(s), 4),
                "terminos_comunes": comunes[pos],
            })
        return pd.DataFrame(filas)

    def recomendar(self, idx, n=5):
        top, sc = self.indices_similares(idx, n)
        return self._tabla(top, sc, [self.terminos_comunes(idx, j) for j in top])

    # ------------------------------------------------------------ consulta por texto libre
    def recomendar_texto(self, texto, n=5):
        limpio = self.pre.transformar([texto])[0]
        q = self.vec.transform([limpio])
        if q.nnz == 0:
            return pd.DataFrame(columns=["id", "titulo", "autor", "generos", "score", "terminos_comunes"])
        sims = linear_kernel(q, self.X).ravel()
        n = min(n, len(sims))
        top = np.argpartition(-sims, n - 1)[:n]
        top = top[np.argsort(-sims[top])]
        comunes = []
        for j in top:
            prod = q.multiply(self.X[j]).tocoo()
            orden = np.argsort(-prod.data)[:5] if prod.nnz else []
            comunes.append([str(self.terminos[prod.col[o]]) for o in orden])
        return self._tabla(top, sims[top], comunes)

    def buscar_titulo(self, texto, limite=10):
        q = normalizar_titulo(texto)
        if not q:
            return self.df.iloc[0:0][["id", "titulo", "autor"]]
        mask = pd.Series(self.titulo_norm).str.contains(re.escape(q), regex=True).values
        return self.df.loc[mask, ["id", "titulo", "autor"]].head(limite)

    # ------------------------------------------------------------ persistencia
    def guardar(self, ruta):
        joblib.dump({"idioma": self.idioma, "bilingue": self.bilingue, "params": self.params, "vec": self.vec,
                     "X": self.X, "df": self.df}, ruta, compress=3)

    @classmethod
    def cargar(cls, ruta):
        d = joblib.load(ruta)
        r = cls(d["idioma"], bilingue=d.get("bilingue", True), **d["params"])
        r.vec, r.X, r.df = d["vec"], d["X"], d["df"]
        r.terminos = r.vec.get_feature_names_out()
        r.titulo_norm = r.df["titulo"].map(normalizar_titulo).values
        return r

In [ ]:
import importlib, recomendador
importlib.reload(recomendador)
from recomendador import Preprocesador, Recomendador

pre = Preprocesador(idioma_texto, bilingue=STOPWORDS_BILINGUES)

# Texto que se vectoriza: título (+ géneros) + descripción
partes = []
if TEXTO_INCLUYE_TITULO:
    partes.append(df["titulo"])
if TEXTO_INCLUYE_GENEROS:
    partes.append(df["generos"])
partes.append(df["descripcion"])
df["texto_original"] = partes[0].str.cat(partes[1:], sep=". ") if len(partes) > 1 else partes[0]

# Ejemplo antes/después
for i in range(3):
    print(f"[{i}] ORIGINAL : {df.loc[i, 'texto_original'][:230]}...")
    print(f"    LIMPIO   : {pre.transformar([df.loc[i, 'texto_original']])[0][:230]}...\n")

In [ ]:
t0 = time.time()
df["texto_limpio"] = pre.transformar(df["texto_original"].tolist(), batch_size=64)
print(f"Preprocesados {len(df):,} textos en {time.time()-t0:.0f} s")
vacios = (df["texto_limpio"].str.len() == 0).sum()
print(f"Textos que quedaron vacíos: {vacios}")
df = df[df["texto_limpio"].str.len() > 0].reset_index(drop=True)
df["id"] = df.index
print(f"Catálogo final: {len(df):,} libros")

## 4. Representación TF-IDF y similitud coseno

### 4.1 Ajuste de `min_df`
`min_df` descarta términos muy raros (errores tipográficos, nombres propios que aparecen una sola vez). Un `min_df` alto reduce el vocabulario y el ruido, pero también elimina términos distintivos. La tabla muestra el efecto; la columna de Precision@K usa la métrica proxy por géneros (sección 5.1), por lo que solo está disponible si el dataset trae géneros.

In [ ]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0

G = df["generos_lista"].tolist()
tiene_generos = sum(len(g) > 0 for g in G) > 0.5 * len(G)
print("Géneros disponibles para la métrica proxy:", tiene_generos)

# Consultas fijas para la métrica proxy: solo libros con géneros
elegibles = [i for i in range(len(df)) if len(G[i]) > 0]
consultas_proxy = rng.choice(elegibles, size=min(N_CONSULTAS_PROXY, len(elegibles)), replace=False) if tiene_generos else []

# Relevantes totales por consulta (para el nDCG ideal)
def relevantes_totales(i):
    return sum(1 for j in range(len(G)) if j != i and jaccard(G[i], G[j]) >= UMBRAL_JACCARD)

def metricas_proxy(fn_top, consultas, k=K):
    """fn_top(i, k) -> lista de índices recomendados. Relevancia binaria por Jaccard de géneros."""
    precs, ndcgs = [], []
    for i in consultas:
        recs = fn_top(i, k)
        rel = [1 if jaccard(G[i], G[j]) >= UMBRAL_JACCARD else 0 for j in recs]
        precs.append(sum(rel) / k)
        dcg = sum(r / np.log2(p + 2) for p, r in enumerate(rel))
        idcg = sum(1 / np.log2(p + 2) for p in range(min(k, relevantes_totales(i))))
        if idcg > 0:
            ndcgs.append(dcg / idcg)
    return float(np.mean(precs)), float(np.mean(ndcgs)) if ndcgs else float("nan")

filas = []
for md_ in [1, 2, 3, 5, 10]:
    r_ = Recomendador(idioma_texto, NGRAMAS, md_, MAX_DF, bilingue=STOPWORDS_BILINGUES).fit(df)
    fila = {"min_df": md_, "vocabulario": len(r_.terminos), "términos/doc (media)": round(r_.X.getnnz(axis=1).mean(), 1)}
    if tiene_generos:
        p, n_ = metricas_proxy(lambda i, k, r_=r_: r_.indices_similares(i, k)[0], consultas_proxy[:150])
        fila.update({f"Precision@{K}": round(p, 3), f"nDCG@{K}": round(n_, 3)})
    filas.append(fila)
tabla_mindf = pd.DataFrame(filas)
tabla_mindf

Criterio para elegir `min_df`: el valor más bajo que ya recorta buena parte del vocabulario sin perder calidad en las métricas. Si la tabla muestra que un `min_df` distinto de 2 rinde claramente mejor, cámbialo en la celda de configuración (sección 0) y vuelve a ejecutar desde aquí.

### 4.2 Modelo final

In [ ]:
rec = Recomendador(idioma_texto, NGRAMAS, MIN_DF, MAX_DF, bilingue=STOPWORDS_BILINGUES).fit(df)
print(f"Matriz TF-IDF: {rec.X.shape[0]:,} libros x {rec.X.shape[1]:,} términos  |  densidad {rec.X.nnz/np.prod(rec.X.shape):.2%}")

# Términos con mayor IDF (más distintivos) y con menor IDF (más comunes)
idf = pd.Series(rec.vec.idf_, index=rec.terminos).sort_values()
print("\nMás comunes :", ", ".join(idf.head(12).index))
print("Más raros   :", ", ".join(idf.tail(12).index))

rec.guardar("artefactos.joblib")
print("\nGuardado artefactos.joblib (lo usan la API y Streamlit)")

### 4.3 Recomendaciones de ejemplo

In [ ]:
ejemplo = int(consultas_proxy[0]) if len(consultas_proxy) else 0
print("LIBRO DE CONSULTA:", df.loc[ejemplo, "titulo"], "|", df.loc[ejemplo, "autor"])
print("Géneros:", df.loc[ejemplo, "generos"])
print("Sinopsis:", df.loc[ejemplo, "descripcion"][:300], "...\n")
rec.recomendar(ejemplo, n=5)

In [ ]:
# Consulta por texto libre
consulta_demo = ("libro de misterio y detectives en una ciudad antigua" if idioma_texto == "es"
                 else "mystery novel with a detective in an old city")
print("Consulta:", consulta_demo)
rec.recomendar_texto(consulta_demo, n=5)

## 5. Evaluación

Se usan **dos evaluaciones complementarias**:

1. **Métrica proxy automática** (5.1): usa los géneros como "juicio" de relevancia. Sirve para comparar configuraciones rápido, pero es un sustituto imperfecto: no mide si al lector le gustaría el libro.
2. **Juicios de usuarios reales** (5.2): es lo que pide el entregable. El notebook genera la encuesta; **la tienes que aplicar tú con compañeros** y volver a cargar los resultados.

### 5.1 Métrica proxy por géneros
Se compara TF-IDF contra una **línea base aleatoria**.

In [ ]:
def top_aleatorio(i, k):
    cand = rng.choice(len(df), size=k + 1, replace=False)
    return [j for j in cand if j != i][:k]

resultados = {}
if tiene_generos:
    resultados["Aleatorio"] = metricas_proxy(top_aleatorio, consultas_proxy)
    resultados["TF-IDF + coseno"] = metricas_proxy(lambda i, k: rec.indices_similares(i, k)[0], consultas_proxy)
    res_df = pd.DataFrame(resultados, index=[f"Precision@{K}", f"nDCG@{K}"]).T.round(3)
    display(res_df)
    res_df.plot.bar(rot=0, figsize=(6, 3.5), color=["#2a7f9e", "#e0a030"])
    plt.title(f"Métrica proxy por géneros (Jaccard >= {UMBRAL_JACCARD}, {len(consultas_proxy)} consultas)")
    plt.ylim(0, 1); plt.tight_layout(); plt.show()
else:
    print("El dataset no trae géneros utilizables: se omite la métrica proxy y se usa solo la encuesta (5.2).")

### 5.2 Encuesta con usuarios reales

> **En la demo esta parte es opcional:** la celda solo genera las plantillas. La sección 5.3 se salta sola mientras no exista `juicios.csv`.

**Procedimiento**
1. Ejecuta la celda siguiente: genera `juicios_plantilla.csv` (para los evaluadores) y `juicios_clave.csv` (**interno**, indica qué filas vienen de TF-IDF y cuáles son aleatorias).
2. Súbela a una hoja de Google Sheets y compártela con **10-15 compañeros**. Cada uno rellena `evaluador` y `calificacion` (1 = nada relacionado, 5 = muy parecido / lo leería después de este libro).
3. Descarga el resultado como `juicios.csv` (con las columnas `fila_id`, `evaluador`, `calificacion`), súbelo al entorno y ejecuta la celda 5.3.

Las recomendaciones aleatorias van **mezcladas a ciegas** entre las de TF-IDF para medir si los evaluadores realmente distinguen el sistema de una línea base.

In [ ]:
consultas_enc = rng.choice(len(df), size=min(N_CONSULTAS_ENCUESTA, len(df)), replace=False)
filas = []
for q in consultas_enc:
    top, _ = rec.indices_similares(q, K)
    excluidos = set(top.tolist()) | {int(q)}
    pool = [i for i in range(len(df)) if i not in excluidos]
    alea = rng.choice(pool, size=3, replace=False)
    for sistema, lista in (("tfidf", top), ("aleatorio", alea)):
        for pos, j in enumerate(lista, 1):
            filas.append({
                "consulta_id": int(q), "rec_id": int(j), "sistema": sistema, "posicion": pos,
                "titulo_consulta": df.loc[q, "titulo"], "sinopsis_consulta": df.loc[q, "descripcion"][:260],
                "titulo_recomendado": df.loc[j, "titulo"], "sinopsis_recomendada": df.loc[j, "descripcion"][:260],
            })
enc = pd.DataFrame(filas)
# Mezcla dentro de cada consulta para que el evaluador no vea el orden del sistema
enc["_azar"] = rng.random(len(enc))
enc = enc.sort_values(["consulta_id", "_azar"]).drop(columns="_azar").reset_index(drop=True)
enc.insert(0, "fila_id", enc.index)

plantilla = enc[["fila_id", "titulo_consulta", "sinopsis_consulta", "titulo_recomendado", "sinopsis_recomendada"]].copy()
plantilla["evaluador"] = ""
plantilla["calificacion"] = ""
plantilla.to_csv("juicios_plantilla.csv", index=False)
enc[["fila_id", "consulta_id", "rec_id", "sistema", "posicion"]].to_csv("juicios_clave.csv", index=False)
print(f"{len(plantilla)} filas a calificar por evaluador ({len(consultas_enc)} libros x 8 recomendaciones).")
plantilla.head(8)

### 5.3 Resultados de la encuesta
Se ejecuta cuando exista `juicios.csv`. Métricas: calificación media, % de recomendaciones relevantes (calificación ≥ 4) y nDCG@K usando la calificación como ganancia graduada.

In [ ]:
ruta_j = Path("juicios.csv")
if not ruta_j.exists():
    print("Modo demo: no hay juicios.csv, se omite esta sección. Para la entrega final aplica la encuesta (5.2) y vuelve a ejecutar.")
else:
    juicios = pd.read_csv(ruta_j)
    juicios["calificacion"] = pd.to_numeric(juicios["calificacion"], errors="coerce")
    juicios = juicios.dropna(subset=["calificacion"])
    clave = pd.read_csv("juicios_clave.csv")
    j = juicios.merge(clave, on="fila_id", how="inner")
    j["relevante"] = (j["calificacion"] >= 4).astype(int)
    print(f"Evaluadores: {j['evaluador'].nunique()}  |  calificaciones: {len(j)}")

    resumen = j.groupby("sistema").agg(
        calificacion_media=("calificacion", "mean"),
        pct_relevantes=("relevante", "mean"),
        n=("calificacion", "size")).round(3)
    display(resumen)

    # nDCG@K del ranking de TF-IDF (promedio de evaluadores por fila)
    medias = j[j["sistema"] == "tfidf"].groupby(["consulta_id", "posicion"])["calificacion"].mean().reset_index()
    ndcgs = []
    for _, g in medias.groupby("consulta_id"):
        g = g.sort_values("posicion")
        ganancias = g["calificacion"].values
        dcg = sum(x / np.log2(p + 2) for p, x in enumerate(ganancias))
        idcg = sum(x / np.log2(p + 2) for p, x in enumerate(sorted(ganancias, reverse=True)))
        ndcgs.append(dcg / idcg if idcg else np.nan)
    print(f"\nnDCG@{K} (TF-IDF, ganancia = calificación): {np.nanmean(ndcgs):.3f}")
    print(f"Precision@{K} (TF-IDF, relevante = calificación >= 4): "
          f"{resumen.loc['tfidf', 'pct_relevantes']:.3f}")
    resumen[["calificacion_media", "pct_relevantes"]].plot.bar(rot=0, figsize=(6, 3.5), color=["#2a7f9e", "#e0a030"])
    plt.title("Juicios de usuarios reales"); plt.tight_layout(); plt.show()

> **Nota para el informe:** el nDCG de la encuesta compara el orden de TF-IDF con el mejor orden posible *entre los mismos cinco libros recomendados*; no mide qué libros se quedaron fuera del top-K. Con pocos evaluadores los intervalos de confianza son amplios, así que conviene reportar el número de evaluadores y no sobreinterpretar diferencias pequeñas.

## 6. Endpoint REST con FastAPI (`/recomendar`)

La API carga `artefactos.joblib` y expone:

| Método | Ruta | Descripción |
|---|---|---|
| GET | `/health` | Estado y tamaño del catálogo |
| GET | `/items?q=texto&limite=10` | Busca libros por título para obtener su `id` |
| GET | `/recomendar?item_id=12&n=5` | **Libros parecidos a uno dado** |
| GET | `/recomendar_texto?texto=...&n=5` | Libros para una descripción en texto libre |

Cada recomendación incluye `score` (coseno) y `terminos_comunes` (explicabilidad).

In [ ]:
%%writefile api.py
from fastapi import FastAPI, HTTPException, Query

from recomendador import Recomendador

app = FastAPI(title="Recomendador de libros (TF-IDF + coseno)", version="1.0")
rec = Recomendador.cargar("artefactos.joblib")


@app.get("/health")
def health():
    return {"estado": "ok", "libros": int(rec.X.shape[0]), "terminos": int(rec.X.shape[1]), "idioma": rec.idioma}


@app.get("/items")
def items(q: str = Query(..., min_length=2), limite: int = Query(10, ge=1, le=50)):
    return rec.buscar_titulo(q, limite).to_dict(orient="records")


@app.get("/recomendar")
def recomendar(item_id: int = Query(..., ge=0), n: int = Query(5, ge=1, le=20)):
    if item_id >= len(rec.df):
        raise HTTPException(status_code=404, detail=f"item_id {item_id} fuera del catálogo (0-{len(rec.df)-1})")
    base = rec.df.loc[item_id]
    return {
        "consulta": {"id": item_id, "titulo": base["titulo"], "autor": base.get("autor", "")},
        "recomendaciones": rec.recomendar(item_id, n).to_dict(orient="records"),
    }


@app.get("/recomendar_texto")
def recomendar_texto(texto: str = Query(..., min_length=3), n: int = Query(5, ge=1, le=20)):
    out = rec.recomendar_texto(texto, n)
    if out.empty:
        raise HTTPException(status_code=422, detail="Ninguna palabra de la consulta está en el vocabulario")
    return {"consulta": texto, "recomendaciones": out.to_dict(orient="records")}

In [ ]:
import uvicorn, importlib, api
importlib.reload(api)

config = uvicorn.Config(api.app, host="127.0.0.1", port=8000, log_level="warning")
servidor = uvicorn.Server(config)
hilo = threading.Thread(target=servidor.run, daemon=True)
hilo.start()

BASE = "http://127.0.0.1:8000"
for _ in range(50):
    try:
        if requests.get(f"{BASE}/health", timeout=1).ok:
            break
    except requests.RequestException:
        time.sleep(0.2)
print("API arriba:", requests.get(f"{BASE}/health").json())

In [ ]:
import json
# Prueba de extremo a extremo
ej_id = int(consultas_proxy[0]) if len(consultas_proxy) else 0
t0 = time.time()
resp = requests.get(f"{BASE}/recomendar", params={"item_id": ej_id, "n": 3})
lat_ms = (time.time() - t0) * 1000
print(f"GET /recomendar?item_id={ej_id}&n=3  ->  {resp.status_code}  ({lat_ms:.0f} ms)")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False)[:1500])

In [ ]:
# Casos de error y búsqueda por título
print("item fuera de rango:", requests.get(f"{BASE}/recomendar", params={"item_id": 10**6}).status_code)
palabra = max(df.loc[ej_id, "titulo"].split(), key=len)  # la API exige al menos 2 caracteres
print(f"/items?q={palabra}:", requests.get(f"{BASE}/items", params={"q": palabra, "limite": 3}).json())

# Latencia sobre 100 llamadas
ids = rng.choice(len(df), 100, replace=False)
lat = []
for i in ids:
    t0 = time.time(); requests.get(f"{BASE}/recomendar", params={"item_id": int(i), "n": 5}); lat.append((time.time() - t0) * 1000)
print(f"\nLatencia /recomendar (100 llamadas): mediana {np.median(lat):.0f} ms | p95 {np.percentile(lat, 95):.0f} ms")

## 7. Dashboard y aplicación web

### 7.1 Dashboard dentro del notebook (ipywidgets)
Escribe parte de un título, elige el libro y mira las recomendaciones con su puntaje y los términos que los conectan.

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

txt = w.Text(placeholder="Escribe parte del título…", description="Buscar:", layout=w.Layout(width="420px"))
dd = w.Dropdown(options=[], description="Libro:", layout=w.Layout(width="620px"))
sl = w.IntSlider(value=5, min=1, max=10, description="Top-N:")
salida = w.Output()

def al_escribir(cambio):
    res = rec.buscar_titulo(cambio["new"], 30)
    dd.options = [(f"{r.titulo[:70]} — {r.autor[:30]}", int(r.id)) for r in res.itertuples()]

def mostrar(_=None):
    with salida:
        clear_output(wait=True)
        if dd.value is None:
            print("Busca un título y selecciónalo."); return
        base = rec.df.loc[dd.value]
        print(f"Libro: {base['titulo']}  |  {base['autor']}")
        res = rec.recomendar(int(dd.value), sl.value)
        display(res)
        ax = res.iloc[::-1].plot.barh(x="titulo", y="score", legend=False, figsize=(7, 0.45 * len(res) + 1), color="#2a7f9e")
        ax.set_xlabel("similitud coseno"); ax.set_ylabel("")
        plt.tight_layout(); plt.show()

txt.observe(al_escribir, names="value")
dd.observe(mostrar, names="value")
sl.observe(mostrar, names="value")
display(w.VBox([txt, dd, sl, salida]))

# Valor inicial para que el panel no aparezca vacío
txt.value = max(df.loc[ej_id, "titulo"].split(), key=len)

### 7.2 Aplicación web con Streamlit
El archivo `app_streamlit.py` usa el mismo módulo y los mismos artefactos. Para verla:

- **En tu computador:** descarga `recomendador.py`, `artefactos.joblib` y `app_streamlit.py` a una carpeta y ejecuta `streamlit run app_streamlit.py`.
- **En Colab:** ejecuta la celda comentada de más abajo, que abre un túnel público temporal con `localtunnel`.

In [ ]:
%%writefile app_streamlit.py
import streamlit as st

from recomendador import Recomendador

st.set_page_config(page_title="Recomendador de libros", page_icon="📚", layout="wide")


@st.cache_resource
def cargar():
    return Recomendador.cargar("artefactos.joblib")


rec = cargar()
st.title("📚 Recomendador de libros")
st.caption(f"TF-IDF + similitud coseno · {rec.X.shape[0]:,} libros · {rec.X.shape[1]:,} términos")

n = st.sidebar.slider("Cantidad de recomendaciones", 1, 10, 5)
modo = st.sidebar.radio("Buscar por", ["Libro parecido a…", "Descripción libre"])

if modo == "Libro parecido a…":
    q = st.text_input("Parte del título")
    if q:
        res = rec.buscar_titulo(q, 30)
        if res.empty:
            st.warning("Sin coincidencias.")
        else:
            opciones = {f"{r.titulo} — {r.autor}": int(r.id) for r in res.itertuples()}
            elegido = st.selectbox("Elige el libro", list(opciones))
            idx = opciones[elegido]
            st.write(rec.df.loc[idx, "descripcion"][:500])
            out = rec.recomendar(idx, n)
            st.subheader("Recomendaciones")
            st.dataframe(out, width="stretch", hide_index=True)
            st.bar_chart(out.set_index("titulo")["score"])
else:
    texto = st.text_area("Describe qué quieres leer")
    if texto.strip():
        out = rec.recomendar_texto(texto, n)
        if out.empty:
            st.warning("Ninguna palabra de tu descripción está en el vocabulario del catálogo.")
        else:
            st.dataframe(out, width="stretch", hide_index=True)
            st.bar_chart(out.set_index("titulo")["score"])

In [ ]:
# --- Solo Colab: descomenta para abrir la app con un túnel público temporal ---
# !pip install -q streamlit
# !streamlit run app_streamlit.py &>/content/streamlit.log &
# !npx --yes localtunnel --port 8501
# (La contraseña del túnel es la IP pública que imprime:  !curl -s ipv4.icanhazip.com)

## 8. (Opcional) Mejora: embeddings frente a TF-IDF
TF-IDF solo compara **palabras idénticas** (después de lematizar). Si una consulta dice "miedo" y el libro dice "terror", no hay coincidencia. Los embeddings multilingües (`paraphrase-multilingual-MiniLM-L12-v2`) capturan sinónimos y paráfrasis, a cambio de más cómputo y menos explicabilidad.

In [ ]:
if not USAR_EMBEDDINGS:
    print("USAR_EMBEDDINGS = False: sección omitida.")
else:
    from sentence_transformers import SentenceTransformer
    modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    textos_emb = (df["titulo"] + ". " + df["descripcion"]).str[:1000].tolist()  # sin lematizar: el modelo prefiere texto natural
    t0 = time.time()
    E = modelo_emb.encode(textos_emb, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    print(f"Embeddings {E.shape} en {time.time()-t0:.0f} s")

    def top_emb(i, k):
        sims = E @ E[i]
        sims[rec.titulo_norm == rec.titulo_norm[i]] = -1.0
        top = np.argpartition(-sims, k - 1)[:k]
        return top[np.argsort(-sims[top])]

In [ ]:
if USAR_EMBEDDINGS:
    if tiene_generos:
        res3 = {
            "Aleatorio": resultados["Aleatorio"],
            "TF-IDF + coseno": resultados["TF-IDF + coseno"],
            "Embeddings": metricas_proxy(top_emb, consultas_proxy),
        }
        r3 = pd.DataFrame(res3, index=[f"Precision@{K}", f"nDCG@{K}"]).T.round(3)
        display(r3)
        r3.plot.bar(rot=0, figsize=(7, 3.5), color=["#2a7f9e", "#e0a030"])
        plt.title("Métrica proxy por géneros"); plt.ylim(0, 1); plt.tight_layout(); plt.show()

    # Latencia por consulta de texto libre
    q_demo = consulta_demo
    t0 = time.time(); rec.recomendar_texto(q_demo, 5); t_tfidf = (time.time() - t0) * 1000
    t0 = time.time(); modelo_emb.encode([q_demo], normalize_embeddings=True); t_emb = (time.time() - t0) * 1000
    print(f"\nLatencia de una consulta de texto: TF-IDF {t_tfidf:.0f} ms | embeddings {t_emb:.0f} ms (solo codificar)")

In [ ]:
if USAR_EMBEDDINGS:
    # Consultas con vocabulario que probablemente NO coincide con las sinopsis: ahí TF-IDF se queda corto
    if idioma_texto == "es":
        consultas_limite = ["una historia que da mucho miedo", "libro para aprender a ganar dinero", "relato de amor entre jóvenes de familias enemigas"]
    else:
        consultas_limite = ["a story that is really scary", "book about getting rich", "love story between teenagers from feuding families"]
    for qtxt in consultas_limite:
        print("=" * 100); print("CONSULTA:", qtxt)
        a = rec.recomendar_texto(qtxt, 3)
        print("  TF-IDF     :", list(a["titulo"].str[:55]) if len(a) else "(sin resultados: ninguna palabra en el vocabulario)")
        e = modelo_emb.encode([qtxt], normalize_embeddings=True)[0]
        sims = E @ e
        top = np.argsort(-sims)[:3]
        print("  Embeddings :", [df.loc[j, "titulo"][:55] for j in top])

## 9. Conclusiones y lista de entrega

**Qué se construyó**
- Pipeline de NLP reutilizable (`recomendador.py`): lematización con spaCy, sin tildes ni stopwords, TF-IDF (1-2 gramas, `sublinear_tf=True`, `min_df` justificado con tabla) y similitud coseno top-N con términos en común como explicación.
- **Endpoint** `GET /recomendar` (FastAPI) con búsqueda por título y consulta por texto libre.
- **Dashboard** en el notebook y **aplicación web** Streamlit.
- Evaluación: métrica proxy por géneros contra una línea base aleatoria y encuesta ciega con usuarios reales.

**Limitaciones que conviene declarar en el informe**
- TF-IDF no entiende sinónimos ni contexto (sección 8 lo muestra).
- Es *content-based*: solo usa texto, no el comportamiento de los lectores, así que no puede descubrir gustos que cruzan géneros.
- La métrica proxy mide cercanía de géneros, no gusto del lector; los juicios humanos son la evidencia principal.
- El catálogo mezcla sinopsis en español e inglés: se quitan stopwords de ambos idiomas, pero la lematización usa un solo modelo de spaCy, así que el texto del otro idioma se lematiza peor.
- Parte de las sinopsis traen caracteres dañados (`#`); esas palabras se descartan.

**Archivos para entregar:** este notebook, `recomendador.py`, `api.py`, `app_streamlit.py`, `juicios_plantilla.csv`, `juicios.csv` (con las respuestas) y el resumen de resultados de la sección 5.3.